Task: For every image, produce a label image with one connected component per part, and
the number of parts. Any pre-processing is allowed (colour spaces, normalisation, denoising,
morphology. . . ) as long as it is justified.

Implement at least N segmentation methodologies of different families (e.g. histogram thresholding, adaptive thresholding, clustering, statistical pixel classification, region growing / watershed, active contours, edge-based. . . ). Two parameter settings of the same algorithm count as one methodology.
• Use the same post-processing for all methods when you compare them, so that the comparison isolates the segmentation step.
• Evaluate on annotated images with, at least, the precision, accuracy and IoU metrics.
• Decide which method is best for this task and explain why, with statistics.

Questions to address: Which colour plane separates the golden screws from the wood? What breaks
a global threshold? How are touching parts handled? Which errors does each method make
(false positives, merges, splits), and are they acceptable for counting?

# Challenge 1: Segmentation and Counting

## 1: Environment Setup & IO Configuration
In this section the following tasks are performed:
- Initialization of the reference environment
- Configuration of relative directory paths
- Using of a set of images to do the exploration part, in order to avoid overfitting

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage.filters import threshold_local
from skimage.morphology import remove_small_objects, remove_small_holes, opening, disk
from scipy.ndimage import binary_fill_holes

# Define project directories
INPUT_DIR = Path("inputs")
ANNOTATIONS_DIR = Path("annotations")
VALID_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Filter only supported image extensions
image_paths = sorted([p for p in INPUT_DIR.iterdir() if p.suffix.lower() in VALID_EXTS])
labelled_image_paths = sorted([p for p in ANNOTATIONS_DIR.iterdir() if p.suffix.lower() in VALID_EXTS])

print(f"Total images found: {len(image_paths)}")
if not image_paths:
    raise FileNotFoundError(f"No valid image files found in '{INPUT_DIR.resolve()}'.")

In [ ]:
DEV_MODE = True       # Set to False when running tests
DEV_SAMPLE_SIZE = 10   # Quick iteration on just 10 images

if DEV_MODE:

    # Pick a spread of images that covers diferent setups
    #dev_stems = {"009","011","018","112","209","377","257","302","199","154"}
    dev_stems = {"026","027","028","029","040","041","042","061","063","119"}
    active_paths = [p for p in image_paths if any(p.stem.endswith(s) for s in dev_stems)]

    print(f"Running in DEV MODE on {len(active_paths)} images.")
else:
    active_paths = image_paths
    print(f"Running in FULL MODE on all {len(active_paths)} images.")

## 2: Exploratory Data Analysis & Color Space Justification

**Core questions addressed:**
- Which colour channels maximise the contrast between parts and the wooden substrate?
- Why do global thresholding approaches fail on this dataset?

In [ ]:
def inspect_image_channels(img_path: Path):
    # cv2.imread loads in BGR
    bgr = cv2.imread(str(img_path))
    if bgr is None:
        print(f"Error loading {img_path}")
        return

    # Color space conversions
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    fig.suptitle(f"Channel Breakdown: {img_path.name}", fontsize=14, weight="bold")

    # Row 1: RGB components
    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title("Original (RGB)")
    axes[0, 1].imshow(rgb[:, :, 0], cmap="Reds")
    axes[0, 1].set_title("Red Channel")
    axes[0, 2].imshow(rgb[:, :, 1], cmap="Greens")
    axes[0, 2].set_title("Green Channel")
    axes[0, 3].imshow(rgb[:, :, 2], cmap="Blues")
    axes[0, 3].set_title("Blue Channel")

    # Row 2: HSV & Grayscale components
    axes[1, 0].imshow(hsv[:, :, 0], cmap="hsv")
    axes[1, 0].set_title("HSV: Hue (H)")
    axes[1, 1].imshow(hsv[:, :, 1], cmap="gray")
    axes[1, 1].set_title("HSV: Saturation (S)")
    axes[1, 2].imshow(hsv[:, :, 2], cmap="gray")
    axes[1, 2].set_title("HSV: Value (V)")
    axes[1, 3].imshow(gray, cmap="gray")
    axes[1, 3].set_title("Grayscale")

    for ax in axes.ravel():
        ax.axis("off")

    plt.tight_layout()
    plt.show()

# Run inspection on your dev set
for path in active_paths[-2:]: # last two images have golden screws
    inspect_image_channels(path)

A: From the above output, it is clear that the golden screw is best separated by the background in Red, Green and HSV Value channels.

- Hue Channel best for isolating washers/rings (also clearly circular)
- Try Thresholding on the value and saturation channels -> combine the resulting mask


In [ ]:
def analyse_global_threshold_breakdown(img_path):
    bgr = cv2.imread(str(img_path))
    if bgr is None:
        return

    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # Compute global Otsu threshold
    otsu_val, otsu_mask = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Calculate Grayscale and RGB Histograms
    hist_gray = cv2.calcHist([gray], [0], None, [256], [0, 256])

    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    fig.suptitle(f"Global Threshold Analysis: {img_path.name}", fontsize=13, weight="bold")

    # Grayscale Histogram with the computed Otsu line
    axes[0].plot(hist_gray, color="black", linewidth=1.5, label="Grayscale Histogram")
    axes[0].axvline(otsu_val, color="red", linestyle="--", linewidth=2, label=f"Otsu Cutoff (T={int(otsu_val)})")
    axes[0].set_title("Global Intensity Distribution")
    axes[0].set_xlabel("Pixel Intensity")
    axes[0].set_ylabel("Pixel Frequency")
    axes[0].set_xlim([0, 256])
    axes[0].legend(loc="upper right")
    axes[0].grid(True, linestyle=":", alpha=0.6)

    # Direct visual of global thresholding
    axes[1].imshow(otsu_mask, cmap="gray")
    axes[1].set_title(f"Resulting Mask (Otsu T={int(otsu_val)})")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

# Run on sample dev images with gold screws
for path in active_paths[:3]:
    analyse_global_threshold_breakdown(path)

Grayscale with a Global Histogram will not work. The wood background is not uniform and the screws/nuts are different colours.
- The intensity histogram of the grayscale image is unimodal due to non-uniform illumination and dark chips in the OSB wood background. A global cutoff leads to false positives in wood textures and splits thin fasteners.


Methodologies:
- Option 1: Local Thresholding -> window can't be as small than smallest nut
- Option 3: GMM/K-means (note: using GMM AND K-means might not count as two distinct methods as in the example list in the task there is only "clustering" as they are in the same family).
- Option 4: active contours
- Option 5: edge-based methods
###

- Option 3: GMM
- Option 4: K-means

In [ ]:
# 1. Load the image (OpenCV loads as BGR by default)
img_bgr = cv2.imread(active_paths[5])

# 2. Convert to RGB and HSV color spaces
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)

# 3. Split into individual channels
# Note: cv2.split on an RGB image returns (R, G, B) if converted first
R, G, B = cv2.split(img_rgb)
H, S, V = cv2.split(img_hsv)

# Create a list of tuples containing the channel name and the 2D array
channels = [
    ('Red', R),
    ('Green', G),
    ('Blue', B),
    ('Hue (H)', H),
    ('Saturation (S)', S),
    ('Value (V)', V)
]

# 4. Set up a 6x2 plotting grid
fig, axes = plt.subplots(nrows=6, ncols=2, figsize=(12, 20))

for i, (name, ch_data) in enumerate(channels):
    # Left Column: Display the channel as a grayscale image
    # Note: OpenCV Hue goes from 0-179, but we keep vmax=255 to maintain scale visually
    vmax = 179 if name == 'Hue (H)' else 255
    axes[i, 0].imshow(ch_data, cmap='gray', vmin=0, vmax=vmax)
    axes[i, 0].set_title(f"{name} Image")
    axes[i, 0].axis('off')

    # Right Column: Plot the intensity histogram
    axes[i, 1].hist(ch_data.ravel(), bins=256, range=[0, 256], color='black', alpha=0.7)
    axes[i, 1].set_title(f"{name} Histogram")
    axes[i, 1].set_xlabel("Pixel Intensity")
    axes[i, 1].set_ylabel("Count")
    axes[i, 1].set_xlim([0, 256])
    axes[i, 1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# 1. Load the image (OpenCV loads as BGR by default)
#img_bgr = cv2.imread(active_paths[0])

# Load image and convert to LAB color space
img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)

# Split into L*, a*, and b* channels
L, a, b = cv2.split(img_lab)

# Group channels for the plotting loop
channels = [
    ('L* (Lightness)', L),
    ('a* (Green-Red)', a),
    ('b* (Blue-Yellow)', b)
]

# Set up a 3x2 plotting grid
fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(12, 12))

for i, (name, ch_data) in enumerate(channels):
    # Left Column: Grayscale Image
    axes[i, 0].imshow(ch_data, cmap='gray', vmin=0, vmax=255)
    axes[i, 0].set_title(f"{name} Image")
    axes[i, 0].axis('off')

    # Right Column: Intensity Histogram
    axes[i, 1].hist(ch_data.ravel(), bins=256, range=[0, 256], color='black', alpha=0.7)
    axes[i, 1].set_title(f"{name} Histogram")
    axes[i, 1].set_xlabel("Pixel Intensity")
    axes[i, 1].set_ylabel("Count")
    axes[i, 1].set_xlim([0, 256])
    axes[i, 1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### Evaluation - Conclusion
From the above evaluations, the conclusions are:
- Red channel is ideal to pick up the majority of screws and it's very good at separating borders.
- Hue value channel is the best to distinguish the gold from the wood
- Hue saturation channel is the ideal to segment the nuts, as their hexagonal shape and hole are clearly distinguishable from the background

## 3: Shared Post-Processing & Evaluation Framework
To ensure a fair comparison under identical conditions, all 3 segmentation methodologies share the same post-processing pipeline and evaluation metrics.

### Pipeline:
- **Area Filtering:** Removes noise artifacts smaller than standard fastener components.
- **Hole filling:** Fills reflections within solid parts.

### 3.2 Evaluation Metrics & Mathematical Formulation

Given the prediction mask ($\hat{Y}$) and the ground truth mask ($Y$), we quantify segmentation quality using pixel-level confusion matrix components: **True Positives ($TP$)**, **False Positives ($FP$)**, **False Negatives ($FN$)**, and **True Negatives ($TN$)**[cite: 5].

#### 1. Precision
$$\text{Precision} = \frac{TP}{TP + FP}$$
* **Why it is included:** Measures the exactness of the segmentation pipeline. In this challenge, background wood textures (OSB) can easily trigger false detections. A high precision ensures that pixels classified as fasteners genuinely belong to parts and not to wooden background artifacts[cite: 2, 5].

#### 2. Recall (Sensitivity)
$$\text{Recall} = \frac{TP}{TP + FN}$$
* **Why it is included:** Measures the completeness of the segmentation. Thin screw threads, fine points, and dark nut bores are prone to under-segmentation[cite: 2, 5]. Recall quantifies how much of the actual fastener area is correctly captured without leaving out critical structural details.

#### 3. Accuracy
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$
* **Why it is included (Explicitly Required):** Explicitly mandated by the project requirements[cite: 2]. It reflects overall pixel-wise correctness across the entire frame. *(Note: Because background pixels vastly outnumber foreground part pixels, accuracy can be artificially inflated by high $TN$; thus, it is always analyzed alongside IoU and F1-Score).*

#### 4. Intersection over Union (IoU / Jaccard Index)
$$\text{IoU} = \frac{TP}{TP + FP + FN}$$
* **Why it is included (Explicitly Required):** The gold-standard metric for object segmentation[cite: 2]. Unlike standard accuracy, IoU penalizes both over-segmentation ($FP$) and under-segmentation ($FN$) simultaneously, evaluating the geometric overlap quality between predicted masks and ground truth[cite: 2, 5].

#### 5. F1-Score (Dice Coefficient)
$$\text{F1} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2 \cdot TP}{2 \cdot TP + FP + FN}$$
* **Why it is included:** Provides the harmonic mean of Precision and Recall. Since counting and classification downstream depend on balanced part boundaries, F1-Score offers a single robust scalar metric to rank algorithms when there is a tradeoff between precision and recall[cite: 2, 5].

In [ ]:
def compute_metrics(pred: np.ndarray, gt: np.ndarray) -> dict:
    # Computes pixel-level Precision, Recall, Accuracy, IoU, and F1 score.
    TP = np.logical_and(pred, gt).sum()
    FP = np.logical_and(pred, ~gt).sum()
    FN = np.logical_and(~pred, gt).sum()
    TN = np.logical_and(~pred, ~gt).sum()

    eps = 1e-9
    return {
        "TP": TP, "FP": FP, "FN": FN, "TN": TN,
        "precision": TP / (TP + FP + eps),
        "recall":    TP / (TP + FN + eps),
        "accuracy":  (TP + TN) / (TP + TN + FP + FN + eps),
        "IoU":       TP / (TP + FP + FN + eps),
        "F1":        2 * TP / (2 * TP + FP + FN + eps),
    }

def post_process_mask(raw_mask: np.ndarray, min_size: int = 2000, hole_size: int = 250) -> np.ndarray:
    """Applies standardized post-processing across all segmentation algorithms."""
    bool_mask = raw_mask.astype(bool)
    clean_mask = remove_small_objects(bool_mask, min_size=min_size)
    filled_mask = remove_small_holes(clean_mask, area_threshold=hole_size)
    return binary_fill_holes(filled_mask)

In [ ]:
# Extract Red (RGB index 0) and Value (HSV index 2) channels
red_channel = img_rgb[:, :, 2]
value_channel = img_hsv[:, :, 2]

blur_kernel = (15, 15)
red_blurred = cv2.GaussianBlur(red_channel, blur_kernel, 0)
val_blurred = cv2.GaussianBlur(value_channel, blur_kernel, 0)

# Apply local thresholding to both channels
block_size = 151
offset = 20

local_thresh_red = threshold_local(red_blurred, block_size=block_size, offset=offset)
mask_red = (red_blurred < local_thresh_red).astype(np.uint8) * 255

local_thresh_val = threshold_local(val_blurred, block_size=block_size, offset=offset)
mask_val = (val_blurred < local_thresh_val).astype(np.uint8) * 255

# 3. Combine Masks
combined_mask = cv2.bitwise_or(mask_red, mask_val)

# 4. Area-Based Cleanup
# Convert to boolean array as required by scikit-image morphology tools
bool_mask = combined_mask.astype(bool)

# Delete all connected white pixel clusters smaller than 1000 pixels
clean_mask = remove_small_objects(bool_mask, min_size=2000)

# Fill in any small black holes inside the solid white screws
clean_mask = remove_small_holes(clean_mask, area_threshold=50)


# Convert back to 8-bit unsigned integer format for OpenCV
clean_mask_uint8 = (clean_mask * 255).astype(np.uint8)

solid_mask_bool = binary_fill_holes(clean_mask_uint8.astype(bool))
solid_mask = solid_mask_bool.astype(np.uint8) * 255



# --- Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(combined_mask, cmap='gray')
axes[0].set_title("Combined Mask (Pre-Cleanup)")
axes[0].axis('off')

axes[1].imshow(solid_mask, cmap='gray')
axes[1].set_title("Solid Mask")
axes[1].axis('off')

axes[2].imshow(img_bgr[..., ::-1])  # Convert BGR to RGB for display
axes[2].set_title("Original Image")
axes[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
X = 10  # quante immagini processare

blur_kernel = (15, 15)
block_size = 151
offset = 5

def compute_metrics(pred, gt):
    """pred e gt: array booleani della stessa shape."""
    TP = np.logical_and(pred, gt).sum()
    FP = np.logical_and(pred, ~gt).sum()
    FN = np.logical_and(~pred, gt).sum()
    TN = np.logical_and(~pred, ~gt).sum()

    eps = 1e-9
    return {
        "TP": TP, "FP": FP, "FN": FN, "TN": TN,
        "precision": TP / (TP + FP + eps),
        "recall":    TP / (TP + FN + eps),
        "accuracy":  (TP + TN) / (TP + TN + FP + FN),
        "IoU":       TP / (TP + FP + FN + eps),
        "F1":        2 * TP / (2 * TP + FP + FN + eps),
    }

rows = []

for idx, (path, label_path) in enumerate(zip(active_paths[:X], labelled_image_paths[:X])):
    img_bgr = cv2.imread(path)

    # --- Pipeline di segmentazione ---
    img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    b_channel = img_lab[:, :, 2]
    b_blurred = cv2.GaussianBlur(b_channel, blur_kernel, 0)

    local_thresh_b = threshold_local(b_blurred, block_size=block_size, offset=offset)
    mask_b = (b_blurred < local_thresh_b).astype(np.uint8) * 255

    bool_mask = mask_b.astype(bool)
    clean_mask = remove_small_objects(bool_mask, min_size=2500)
    clean_mask_uint8 = (clean_mask * 255).astype(np.uint8)

    close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    closed_mask = cv2.morphologyEx(clean_mask_uint8, cv2.MORPH_CLOSE, close_kernel)

    final_mask_bool = remove_small_holes(closed_mask.astype(bool), area_threshold=250)
    pred_bool = final_mask_bool

    # --- Ground truth (PNG 16 bit, scala di grigi) ---
    gt = cv2.imread(label_path, cv2.IMREAD_UNCHANGED)  # mantiene uint16
    if gt.ndim == 3:  # sicurezza: se per caso ha più canali
        gt = gt[:, :, 0]
    if gt.shape != pred_bool.shape:
        gt = cv2.resize(gt, (pred_bool.shape[1], pred_bool.shape[0]),
                        interpolation=cv2.INTER_NEAREST)
    gt_bool = gt > 0  # qualsiasi valore > 0 = oggetto (vite/dado)

    # --- Metriche ---
    m = compute_metrics(pred_bool, gt_bool)
    m["image"] = idx
    rows.append(m)

    # --- Visualizzazione: verde = TP, rosso = FP, blu = FN ---
    overlay = np.zeros((*pred_bool.shape, 3), dtype=np.uint8)
    overlay[pred_bool & gt_bool]  = (0, 255, 0)
    overlay[pred_bool & ~gt_bool] = (255, 0, 0)
    overlay[~pred_bool & gt_bool] = (0, 0, 255)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); axes[0].set_title(f"Originale [{idx}]")
    axes[1].imshow(gt_bool, cmap='gray');                     axes[1].set_title("Label")
    axes[2].imshow(overlay)
    axes[2].set_title(f"IoU={m['IoU']:.3f}  P={m['precision']:.3f}  Acc={m['accuracy']:.3f}")
    for ax in axes: ax.axis('off')
    plt.show()

# --- Tabella riassuntiva ---
df = pd.DataFrame(rows).set_index("image")
display(df[["precision", "recall", "accuracy", "IoU", "F1"]].round(4))

print("Media per immagine:")
print(df[["precision", "recall", "accuracy", "IoU", "F1"]].mean().round(4))

# Metriche globali (pixel aggregati su tutte le immagini)
tot = df[["TP", "FP", "FN", "TN"]].sum()
print("\nMetriche globali (pixel aggregati):")
print(f"Precision: {tot.TP / (tot.TP + tot.FP):.4f}")
print(f"Accuracy:  {(tot.TP + tot.TN) / tot.sum():.4f}")
print(f"IoU:       {tot.TP / (tot.TP + tot.FP + tot.FN):.4f}")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture

# Load image and use RGB (or substitute with your LAB/HSV stack)
img_bgr = cv2.imread(active_paths[1])
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# Flatten the image into a 2D array of pixels: (Height * Width, 3 channels)
# Subsampling the pixels speeds up the GMM fitting process significantly
pixels = img_rgb.reshape(-1, 3)

# Initialize GMM with 3 components (Wood, Dark Metal, Bright Metal)
gmm = GaussianMixture(n_components=2, covariance_type='tied', random_state=13)

# Fit the model and predict the cluster label (0, 1, or 2) for each pixel
labels = gmm.fit_predict(pixels)

# Reshape the 1D labels back to the original 2D image dimensions
segmented_image = labels.reshape(img_rgb.shape[0], img_rgb.shape[1])

# --- Visualization ---
plt.imshow(segmented_image, cmap='viridis')
plt.title("GMM Pixel Clustering (2 Components)")
plt.axis('off')
plt.show()

In [ ]:
import numpy as np, cv2
import matplotlib.pyplot as plt
from skimage import color, filters, morphology, measure
from skimage.segmentation import mark_boundaries

img_bgr = cv2.imread(active_paths[7])
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

lab = color.rgb2lab(img_rgb)
hsv = color.rgb2hsv(img_rgb)
chans = {"L*": lab[...,0], "a*": lab[...,1], "b*": lab[...,2], "Saturation": hsv[...,1]}

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
axes[0].imshow(img_rgb); axes[0].set_title("Original")
for ax, (n, c) in zip(axes[1:], chans.items()):
    im = ax.imshow(c, cmap="gray"); ax.set_title(n); plt.colorbar(im, ax=ax, fraction=0.046)
for ax in axes: ax.axis("off")
plt.show()

In [ ]:
def seg_tophat(rgb, channel=2, R=15, sigma=1.0, low_factor=0.5, min_area_frac=3e-4, return_all=False):
    lab = color.rgb2lab(rgb)
    c = filters.gaussian(lab[..., channel], sigma)          # light smoothing only
    th = morphology.black_tophat(c, morphology.disk(R))     # dark things narrower than 2R
    high = filters.threshold_otsu(th)
    m = filters.apply_hysteresis_threshold(th, low_factor * high, high)
    min_area = int(min_area_frac * m.size)                  # relative to image size
    m = morphology.remove_small_objects(m, min_area)
    m = morphology.remove_small_holes(m, min_area // 3)
    return (m, th, high) if return_all else m

mask, th, high = seg_tophat(img_rgb, return_all=True)
n = measure.label(mask).max()

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
axes[0].imshow(th, cmap="magma"); axes[0].set_title("b* black top-hat")
axes[1].hist(th.ravel(), bins=100, log=True); axes[1].axvline(high, color="r"); axes[1].set_title("histogram (log)")
axes[2].imshow(mask, cmap="gray"); axes[2].set_title(f"mask: {n} components")
axes[3].imshow(mark_boundaries(img_rgb, mask, color=(1,0,0), mode="thick")); axes[3].set_title("overlay")
for i in (0, 2, 3): axes[i].axis("off")
plt.show()
